# Advanced RAG Pipeline with Reranking
## Azerbaijan National AI Strategy 2025–2028

This notebook implements a full **Retrieval-Augmented Generation (RAG)** pipeline with a reranking step,
applied to the official *AR National Strategy for Artificial Intelligence 2025–2028* (Sections 1–7).

**Pipeline:**
1. **Phase 1** — Data Ingestion & Chunking
2. **Phase 2** — Vector Store & Embedding (FAISS + multilingual MiniLM)
3. **Phase 3** — Reranking with Cross-Encoder
4. **Phase 4** — LLM Generation in Azerbaijani (via Groq API)

---
> **Note:** You need a free [Groq API key](https://console.groq.com) to run Phase 4.
> The PDF of the AI Strategy document must be uploaded when prompted.


### Install Required Libraries

In [44]:
# import subprocess
# import sys

# packages = [
#     "numpy<2.0",     
#     "pdfplumber",
#     "faiss-cpu",
#     "sentence-transformers",
#     "groq",
#     "ipywidgets",
#     "ipython"
# ]

# for package in packages:
#     print(f"Installing {package}...")
#     subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# print("All packages installed successfully.")

## Phase 0: Import Libraries

In [45]:
import pdfplumber
import re
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from groq import Groq
import ipywidgets as widgets
from IPython.display import display, clear_output 

print("All libraries imported successfully")


All libraries imported successfully


## Phase 1:

### Data Ingestion — Upload & Extract PDF

Upload the **AR National Strategy for Artificial Intelligence 2025–2028** PDF.  
Text is extracted from every page using `pdfplumber`.


In [46]:
pdf_filename = r'C:\Users\USER\Documents\PYTHON\MYDL\RAG Pipeline with Reranking\2528 strategiya.pdf'
print(f"Running locally.\nUsing file: {pdf_filename}")

with pdfplumber.open(pdf_filename) as pdf:
    full_text = ""
    for page in pdf.pages:
        text = page.extract_text()
        if text:
            full_text += text + "\n"

print(f"\n Total characters extracted: {len(full_text):,}")
print("\n  Preview (first 500 chars) \n",full_text[:500])


Running locally.
Using file: C:\Users\USER\Documents\PYTHON\MYDL\RAG Pipeline with Reranking\2528 strategiya.pdf

 Total characters extracted: 44,278

  Preview (first 500 chars) 
 “Azərbaycan Respublikasının 2025–2028-ci illər üçün süni intellekt Strategiyası”nın təsdiq edilməsi haqqında
AZƏRBAYCAN RESPUBLİKASI PREZİDENTİNİN SƏRƏNCAMI
Azərbaycan Respublikası Konstitusiyasının 109-cu maddəsinin 3-cü və 32-ci bəndlərini rəhbər tutaraq, Azərbaycan
Respublikasında süni intellektin inkişafını sürətləndirmək, bu sahədə tədqiqatların aparılmasını təşviq etmək, süni
intellekt üzrə informasiya texnologiyalarının və onların idarə edilməsi mexanizmlərinin təkmilləşdirilməsini,
infra


### Truncate Text to Sections 1–7

We locate all occurrences of `\n8.` in the text, then keep only everything **before**
the second match (the first match may appear in a table of contents).


In [47]:
target_line = r'8\. Strategiyanın həyata keçirilməsi üzrə Tədbirlər Planı'

match = re.search(target_line, full_text)
if match:
    cutoff_idx = match.start()
    text_sections_1_7 = full_text[:cutoff_idx]  
else:
    text_sections_1_7 = full_text  

print(f"Sections 1–7 length: {len(text_sections_1_7):,} characters")
print("\n--- Last 300 chars of Section 7 ---")
print(text_sections_1_7[-300:])



Sections 1–7 length: 35,796 characters

--- Last 300 chars of Section 7 ---
keçirilməsi üzrə Tədbirlər Planında qeyd olunan tədbirlərin maliyyələşdirilməsi büdcə təşkilatları
tərəfindən Azərbaycan Respublikasının dövlət büdcəsində nəzərdə tutulan vəsait və qanunla qadağan olunmayan digər mənbələr
hesabına, habelə dövlət–özəl tərəfdaşlığı layihələri vasitəsilə təmin olunur.



### Chunking

Text is split into **500-character chunks** with a **50-character overlap** so that
semantic context is preserved across chunk boundaries.


In [48]:
CHUNK_SIZE = 500
OVERLAP    = 50

chunks = []
start = 0
while start < len(text_sections_1_7):
    end = start + CHUNK_SIZE
    chunks.append(text_sections_1_7[start:end])
    start += CHUNK_SIZE - OVERLAP 

print(f"\n Total chunks created: {len(chunks)}")
print(f"   Chunk size  : {CHUNK_SIZE} chars")
print(f"   Overlap     : {OVERLAP} chars")

print("\n--- Example: First Chunk ---")
print(chunks[0])

print("\n--- Example: Second Chunk (shows overlap) ---")
print(chunks[1])



 Total chunks created: 80
   Chunk size  : 500 chars
   Overlap     : 50 chars

--- Example: First Chunk ---
“Azərbaycan Respublikasının 2025–2028-ci illər üçün süni intellekt Strategiyası”nın təsdiq edilməsi haqqında
AZƏRBAYCAN RESPUBLİKASI PREZİDENTİNİN SƏRƏNCAMI
Azərbaycan Respublikası Konstitusiyasının 109-cu maddəsinin 3-cü və 32-ci bəndlərini rəhbər tutaraq, Azərbaycan
Respublikasında süni intellektin inkişafını sürətləndirmək, bu sahədə tədqiqatların aparılmasını təşviq etmək, süni
intellekt üzrə informasiya texnologiyalarının və onların idarə edilməsi mexanizmlərinin təkmilləşdirilməsini,
infra

--- Example: Second Chunk (shows overlap) ---
ilməsi mexanizmlərinin təkmilləşdirilməsini,
infrastrukturunun əlçatanlığını təmin etmək, habelə ixtisaslı kadr potensialını gücləndirmək məqsədilə qərara
alıram:
1. “Azərbaycan Respublikasının 2025–2028-ci illər üçün süni intellekt Strategiyası” (bundan sonra – Strategiya)
təsdiq edilsin (əlavə olunur).
2. Azərbaycan Respublikasının Nazirl

## Phase 2

### Embedding

Each chunk is converted into a dense vector using a **multilingual sentence-transformer** model.  
Model: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`  
This model supports 50+ languages, including Azerbaijani.


In [49]:
EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
print(f"Loading embedding model: {EMBED_MODEL_NAME} ...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

print("Encoding all chunks ...")
embeddings = embed_model.encode(chunks, convert_to_numpy=True, show_progress_bar=True)

print(f"\nEmbedding matrix shape: {embeddings.shape}")
print(f"   {embeddings.shape[0]} chunks  ×  {embeddings.shape[1]} dimensions")

Loading embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding all chunks ...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


Embedding matrix shape: (80, 384)
   80 chunks  ×  384 dimensions


### FAISS Vector Store & Similarity Search

All embeddings are stored in a **FAISS** flat L2 index.  
We query the index with the user's question and retrieve the **top-10** most similar chunks.

> **Query (Azerbaijani):** *Süni intellekt sahəsində insan resurslarının inkişafına dair strateji məqsədlər nələrdir?*  
> *(What are the strategic goals for human resource development in AI?)*


In [50]:
import faiss

dimension = embeddings.shape[1]
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(embeddings)
print(f"FAISS index built with {faiss_index.ntotal} vectors.")


QUERY = input("Sualınızı yazın (query): ")

query_embedding = embed_model.encode([QUERY], convert_to_numpy=True)

distances, indices = faiss_index.search(query_embedding, 10)
top10_chunks = [chunks[i] for i in indices[0]]

print("\n--- Top-10 Retrieved Chunks (preview) ---")
for rank, (chunk, dist) in enumerate(zip(top10_chunks, distances[0]), 1):
    print(f"\n[Rank {rank} | Cosine sim: {dist:.4f}]")
    print(chunk[:150] + "...")

FAISS index built with 80 vectors.

--- Top-10 Retrieved Chunks (preview) ---

[Rank 1 | Cosine sim: 6.4111]
dır:
1. biznes proseslərini avtomatlaşdırmaqla (robotlardan və avtonom nəqliyyat vasitələrindən istifadə daxil olmaqla)
məhsuldarlığın artırılması;
2....

[Rank 2 | Cosine sim: 6.5045]
şdırılması, vətəndaş iştirakçılığının genişləndirilməsi ilə yanaşı, startaplar və digər biznes
subyektlərinin yeni məhsul və xidmətlər yaratmaq, o cüm...

[Rank 3 | Cosine sim: 6.9972]
ir.
Süni intellekt ekosisteminin gücləndirilməsi, süni intellektin tətbiqini genişləndirmək üçün pilot layihələrin icrası və
nəticədə biznesin inkişaf...

[Rank 4 | Cosine sim: 7.0231]
mərkəzlərinin, Süni İntellekt Akademiyasının
və laboratoriyaların inkişaf etdirilməsi, startaplara dəstək proqramlarının təşkil olunması planlaşdırılı...

[Rank 5 | Cosine sim: 7.0247]
arklarında fəaliyyət göstərən rezidentlərin artırılması ölkənin rəqəmsal iqtisadiyyatını gücləndirəcəkdir. Bu
tədbirlər süni intellekt sahəsində innov.

## Phase 3: Reranking with Cross-Encoder

The **Cross-Encoder** jointly processes the (query, chunk) pair and produces a fine-grained
relevance score — more accurate than the bi-encoder similarity used for initial retrieval.

Model: `cross-encoder/ms-marco-MiniLM-L-6-v2`

We keep only the **top-3** highest-scoring chunks to pass to the LLM.


In [51]:
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
print(f" Loading Cross-Encoder: {RERANKER_MODEL} ...")
reranker = CrossEncoder(RERANKER_MODEL)

print(" Reranking top-10 chunks...")
pairs  = [(QUERY, chunk) for chunk in top10_chunks]
scores = reranker.predict(pairs)

ranked_indices = np.argsort(scores)[::-1]
TOP_K = 3
topk_indices = ranked_indices[:TOP_K]
topk_chunks  = [top10_chunks[i] for i in topk_indices]
topk_scores  = [scores[i]       for i in topk_indices]

print("\n Top-3 Chunks After Reranking:")
for rank, (chunk, score) in enumerate(zip(topk_chunks, topk_scores), 1):
    print(f"\n{'='*60}")
    print(f"[Rank {rank} | Rerank Score: {score:.4f}]")
    print(f"{'='*60}")
    print(chunk)


 Loading Cross-Encoder: cross-encoder/ms-marco-MiniLM-L-6-v2 ...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Reranking top-10 chunks...

 Top-3 Chunks After Reranking:

[Rank 1 | Rerank Score: 6.9925]
mərkəzlərinin, Süni İntellekt Akademiyasının
və laboratoriyaların inkişaf etdirilməsi, startaplara dəstək proqramlarının təşkil olunması planlaşdırılır;
6.1.4. Əlverişli biznes mühitinin yaradılması:
6.1.4.1. Əsaslandırma: əlverişli biznes mühitinin yaradılması sahəsində süni intellekt texnologiyalarının tətbiqi üzrə
yerli və xarici investisiyaların cəlb edilməsi, sahibkarlıq subyektlərinə dəstək mexanizmlərinin genişləndirilməsi və
texnologiya parklarında fəaliyyət göstərən rezidentlərin artırı

[Rank 2 | Rerank Score: 4.1466]
dır:
1. biznes proseslərini avtomatlaşdırmaqla (robotlardan və avtonom nəqliyyat vasitələrindən istifadə daxil olmaqla)
məhsuldarlığın artırılması;
2. müəssisələrdə mövcud işçi qüvvəsini süni intellekt texnologiyaları (yardımçı və genişləndirilmiş intellekt) ilə
inkişaf etdirərək məhsuldarlığın artırılması;
3. fərdiləşdirilmiş və (və ya) daha yüksəkkeyfiyyətli süni intel

## Phase 4: LLM Generation

### Step 1 — Set Your Groq API Key

Get a free API key from [console.groq.com](https://console.groq.com).  
Replace `"YOUR_GROQ_API_KEY"` below with your actual key.

> ⚠️ **Security note:** Never commit real API keys to public repositories.
> Use environment variables or secret managers in production.


In [ ]:
import os

GROQ_API_KEY = "your_api" 

if not GROQ_API_KEY or GROQ_API_KEY == "YOUR_GROQ_API_KEY":
        raise ValueError("Please set a valid Groq API key.")
else:
    groq_client = Groq(api_key=GROQ_API_KEY)
    print("Groq client initialized.")


Groq client initialized.


## Phase 4: Generate Answer with LLM

We construct a system prompt that instructs the LLM to:
- Answer **strictly** from the provided context (no hallucinations)
- Respond **in Azerbaijani**


In [53]:
def build_prompt(query: str, context_chunks: list[str]):
    """Build the RAG prompt from top-k chunks and the user query."""
    context = "\n\n".join(context_chunks)
    return f"""Siz Azərbaycan Respublikasının Süni İntelekt Strategiyası üzrə ekspertsiniz.
Aşağıda verilmiş kontekstə əsaslanaraq sualı YALNIZ Azərbaycan dilində cavablandırın.
Kontekstdə olmayan heç bir məlumat əlavə etməyin.

Kontekst:
{context}

Sual: {query}

Cavab:"""


def ask_rag(query: str):
    """Full RAG pipeline: embed → retrieve → rerank → generate."""
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    _, idxs = faiss_index.search(q_emb, 10)
    top10 = [chunks[i] for i in idxs[0]]

    sc = reranker.predict([(query, c) for c in top10])
    bestk =[top10[i] for i in np.argsort(sc)[::-1][:TOP_K]]

    prompt = build_prompt(query, bestk)
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return response.choices[0].message.content


# print(f" Query: {QUERY}\n")
# print(" Generating answer...")
# answer = ask_rag(QUERY)
# print("\n=== LLM CAVABI (LLM RESPONSE) ===")
# print(answer)


## Phase 4: Interactive Q&A Interface

Use the widget below to ask any question about the Azerbaijan AI Strategy.
The full RAG + reranking pipeline runs on every submission.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

question_box = widgets.Textarea(
    placeholder="Sualınızı buraya yazın...",
    layout=widgets.Layout(
        width="700px",
        height="80px",
        padding="10px",
        border="2px solid #4CAF50",
        border_radius="8px",
        overflow_y="auto"
    ),
    style={"font_size": "14px", "font_family": "Arial"}
)

ask_btn = widgets.Button(
    description="Soruş / Ask",
    button_style="success",
    layout=widgets.Layout(width="150px", height="40px"),
    style={"font_weight": "bold", "font_size": "14px"}
)

out_area = widgets.Output(
    layout=widgets.Layout(
        width="700px",
        height="200px",
        border="2px solid #2196F3",
        border_radius="8px",
        padding="10px",
        overflow_y="auto",
        background_color="#f5f5f5"
    )
)

def on_ask(b):
    with out_area:
        clear_output()
        q = question_box.value.strip()
        if not q:
            display(widgets.HTML("<b style='color:red;'>Zəhmət olmasa sualınızı daxil edin.</b>"))
            return
        display(widgets.HTML(f"<b>Sual:</b> {q}<br><i>Cavab hazırlanır...</i><br>"))
        try:
            ans = ask_rag(q)  
            display(widgets.HTML(f"<pre style='white-space: pre-wrap; font-size:14px'>{ans}</pre>"))
        except Exception as e:
            display(widgets.HTML(f"<b style='color:red;'>Xəta:</b> {e}"))

ask_btn.on_click(on_ask)


display(
    widgets.HTML("<h2 style='color:#4CAF50;'>Süni İntellekt Chatbot</h2>"),
    question_box,
    ask_btn,
    out_area
)

HTML(value="<h2 style='color:#4CAF50;'>Süni İntellekt Chatbot</h2>")

Textarea(value='', layout=Layout(border_bottom='2px solid #4CAF50', border_left='2px solid #4CAF50', border_ri…

Button(button_style='success', description='Soruş / Ask', layout=Layout(height='40px', width='150px'), style=B…

Output(layout=Layout(border_bottom='2px solid #2196F3', border_left='2px solid #2196F3', border_right='2px sol…

: 